# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an end-to-end example for discovering and manipulating the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their `@id` values. This information is needed for loading and processing specific records.

In [ ]:
# List available record sets (they correspond to tables or main data objects)
print("Available record sets (accessible via their @id):")
for record_set in dataset.record_sets:
    print(f"  - name: {record_set.name} | @id: {record_set.id}")

# Select the primary record set for this dataset by @id
# (There may be multiple; adjust as necessary. Here we choose the main table if more exist.)
record_set_id = None
for record_set in dataset.record_sets:
    # Pick the main experimental table if possible
    if "experiment" in record_set.name.lower() or "main" in record_set.name.lower():
        record_set_id = record_set.id
        break
if record_set_id is None and len(dataset.record_sets) > 0:
    # Fallback to the first record set
    record_set_id = dataset.record_sets[0].id

print(f"\nSelected record set for exploration: {record_set_id}")

# List fields (columns) in this record set
print("\nFields in the selected record set:")
for field in dataset.record_set(record_set_id).fields:
    print(f"  - name: {field.name} | @id: {field.id} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from the selected record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract all record sets into DataFrames for analysis
dataframes = {}
# Gather all record set ids
record_set_ids = [rs.id for rs in dataset.record_sets]

for rid in record_set_ids:
    print(f"Loading record set: {rid}")
    records = list(dataset.records(record_set=rid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rid] = df

print("\nColumns in the selected record set:")
print(dataframes[record_set_id].columns.tolist())

dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Select a numeric field for analysis (use its @id from the field list above)
# Determined via overview -- below is an example; change variable if appropriate
selected_numeric_field_id = None
for field in dataset.record_set(record_set_id).fields:
    # Here we heuristically pick a likely numeric field, e.g., 'MW' or 'Abundance'
    if field.data_type in ["Float", "Integer", "Number"]:
        selected_numeric_field_id = field.id
        print(f"Using numeric field: {field.name} (@id: {selected_numeric_field_id})")
        break

if selected_numeric_field_id is None:
    raise ValueError("Could not find a numeric field to analyze.")

# Filtering: show records with value greater than a threshold
threshold = 10
df = dataframes[record_set_id]

# Some columns may come in as string, so convert numeric if needed
if not pd.api.types.is_numeric_dtype(df[selected_numeric_field_id]):
    df[selected_numeric_field_id] = pd.to_numeric(df[selected_numeric_field_id], errors='coerce')

filtered_df = df[df[selected_numeric_field_id] > threshold]
print(f"Filtered records with {selected_numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{selected_numeric_field_id}_normalized"] = (
    filtered_df[selected_numeric_field_id] - filtered_df[selected_numeric_field_id].mean()
) / filtered_df[selected_numeric_field_id].std()

print(f"\nNormalized {selected_numeric_field_id} for filtered records:")
print(filtered_df[[selected_numeric_field_id, f"{selected_numeric_field_id}_normalized"]].head())

# Group by a categorical field if one exists
group_field_id = None
for field in dataset.record_set(record_set_id).fields:
    # Example: group by 'description', 'group', or other non-numeric field
    if field.data_type not in ["Float", "Integer", "Number"]:
        group_field_id = field.id
        print(f"Grouping by: {field.name} (@id: {group_field_id})")
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[selected_numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean of {selected_numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[selected_numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {selected_numeric_field_id} (> {threshold})')
plt.xlabel(selected_numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[selected_numeric_field_id])
    plt.title(f'{selected_numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=60, ha='right')
    plt.show()

## 6. Conclusion
We have loaded and explored the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant`, examined record set and field structure via their `@id`s, and performed basic exploratory analysis and visualizations. You can adapt the data manipulation and EDA examples above for your own scientific questions and downstream ML workflows.